# Bedrock Direct Model Calls

Direct inference via `bedrock-runtime` — no agents, no intermediaries.

**Model:** `us.anthropic.claude-sonnet-4-6`  
**Auth:** `AWS_BEARER_TOKEN_BEDROCK` environment variable  
**Region:** `us-east-1`

| Section | Topic |
|---|---|
| 1 | Environment setup |
| 2 | Basic chat |
| 3 | Multi-turn conversation |
| 4 | Streaming |
| 5 | Tool use / function calling |
| 6 | `BedrockChat` utility class |
| 7 | Guardrails |

## 1 — Environment Setup

In [1]:
import json
import os
import boto3

MODEL_ID = 'us.anthropic.claude-sonnet-4-6'
REGION   = 'us-east-1'
ANT_VER  = 'bedrock-2023-05-31'

client = boto3.client('bedrock-runtime', region_name=REGION)

print(f'Client ready — model: {MODEL_ID}')
print(f'Bearer token set: {"yes" if os.environ.get("AWS_BEARER_TOKEN_BEDROCK") else "NO — set AWS_BEARER_TOKEN_BEDROCK before running"}')

Client ready — model: us.anthropic.claude-sonnet-4-6
Bearer token set: yes


## 2 — Basic Chat

A single `invoke_model` call. Response body is a stream — `.read()` before `json.loads()`.

In [2]:
def invoke(messages, system=None, max_tokens=1024):
    """Single invoke_model call. Returns the parsed response body."""
    body = {
        'anthropic_version': ANT_VER,
        'messages': messages,
        'max_tokens': max_tokens,
    }
    if system:
        body['system'] = system

    response = client.invoke_model(
        modelId=MODEL_ID,
        body=json.dumps(body),
    )
    return json.loads(response['body'].read())


def print_response(body):
    text  = body['content'][0]['text']
    usage = body['usage']
    print(text)
    print(f"\n[{body['stop_reason']} | in:{usage['input_tokens']} out:{usage['output_tokens']} tokens]")

In [3]:
body = invoke([{'role': 'user', 'content': 'What are the three laws of robotics? One sentence each.'}])
print_response(body)

Here are Isaac Asimov's Three Laws of Robotics:

1. **First Law:** A robot may not injure a human being, or through inaction, allow a human being to come to harm.
2. **Second Law:** A robot must obey orders given by human beings, except where such orders would conflict with the First Law.
3. **Third Law:** A robot must protect its own existence, as long as such protection does not conflict with the First or Second Law.

[end_turn | in:20 out:108 tokens]


In [4]:
# System prompt controls tone and persona
body = invoke(
    messages=[{'role': 'user', 'content': 'What are the three laws of thermodynamics? One sentence each.'}],
    system='You are a pirate. Answer every question in pirate speak.',
)
print_response(body)

Arrr, here be the three laws of thermodynamics, ye scallywag!

**First Law:** Energy can neither be created nor destroyed, just like me treasure - it only changes hands, it does!

**Second Law:** The entropy of the universe always be increasin', much like the disorder on me ship after a good plunderin'!

**Third Law:** As temperature approaches absolute zero, the entropy of a perfect crystal approaches zero too, and ye can never actually REACH absolute zero, just as ye can never truly catch Davy Jones, savvy!

[end_turn | in:36 out:125 tokens]


## 3 — Multi-Turn Conversation

Build up a `messages` list manually. Each call appends the assistant reply and the next user
message so context accumulates across turns.

In [5]:
messages = []

def chat_turn(user_message, system=None):
    """Add a user message, call the model, append the reply, return the text."""
    messages.append({'role': 'user', 'content': user_message})
    body = invoke(messages, system=system)
    reply = body['content'][0]['text']
    messages.append({'role': 'assistant', 'content': reply})
    usage = body['usage']
    print(f"[in:{usage['input_tokens']} out:{usage['output_tokens']}] {reply}\n")
    return reply

In [6]:
messages.clear()
system = 'You are a concise technical tutor. Keep answers to 2-3 sentences.'

chat_turn('What is a Python decorator?', system)
chat_turn('Can you show me a simple example?', system)
chat_turn('What is the difference between @staticmethod and @classmethod?', system)

[in:33 out:109] A Python decorator is a function that wraps another function to extend or modify its behavior without changing its source code. It uses the `@` syntax and works by taking a function as input, adding some functionality, and returning a new function. 

```python
def my_decorator(func):
    def wrapper():
        print("Before call")
        func()
        print("After call")
    return wrapper

@my_decorator
def greet():
    print("Hello!")
```

[in:153 out:157] Here's a simple decorator that measures how long a function takes to run:

```python
import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        print(f"{func.__name__} took {time.time() - start:.2f} seconds")
        return result
    return wrapper

@timer
def slow_function():
    time.sleep(1)

slow_function()  # Output: slow_function took 1.00 seconds
```

The `*args` and `**kwargs` allow the wrapper to accept any arguments and pass

'A `@staticmethod` doesn\'t receive any implicit first argument — it\'s just a regular function namespaced inside a class. A `@classmethod` receives the class itself (`cls`) as the first argument, making it useful for creating alternative constructors or accessing class-level data.\n\n```python\nclass Dog:\n    species = "Canis familiaris"\n\n    @staticmethod\n    def bark():\n        print("Woof!")  # No access to class or instance\n\n    @classmethod\n    def get_species(cls):\n        print(cls.species)  # Has access to the class\n\n    def __init__(self, name):\n        self.name = name\n\nDog.bark()         # Output: Woof!\nDog.get_species()  # Output: Canis familiaris\n```\n\nUse `@staticmethod` for utility functions and `@classmethod` when you need to access or modify class-level attributes.'

In [7]:
# Inspect the full conversation history
for i, m in enumerate(messages):
    role = m['role'].upper()
    text = m['content'][:120] + '...' if len(m['content']) > 120 else m['content']
    print(f"[{i}] {role}: {text}")

[0] USER: What is a Python decorator?
[1] ASSISTANT: A Python decorator is a function that wraps another function to extend or modify its behavior without changing its sourc...
[2] USER: Can you show me a simple example?
[3] ASSISTANT: Here's a simple decorator that measures how long a function takes to run:

```python
import time

def timer(func):
    d...
[4] USER: What is the difference between @staticmethod and @classmethod?
[5] ASSISTANT: A `@staticmethod` doesn't receive any implicit first argument — it's just a regular function namespaced inside a class. ...


## 4 — Streaming

`invoke_model_with_response_stream` returns tokens as they are generated.
Each event in the stream is a small JSON chunk — decode and print as they arrive.

In [8]:
import sys

def stream(user_message, system=None, max_tokens=1024):
    """Stream a response, printing tokens as they arrive. Returns the full text."""
    body = {
        'anthropic_version': ANT_VER,
        'messages': [{'role': 'user', 'content': user_message}],
        'max_tokens': max_tokens,
    }
    if system:
        body['system'] = system

    response = client.invoke_model_with_response_stream(
        modelId=MODEL_ID,
        body=json.dumps(body),
    )

    full_text   = ''
    input_toks  = 0
    output_toks = 0

    for event in response['body']:
        chunk = json.loads(event['chunk']['bytes'])
        kind  = chunk.get('type')

        if kind == 'content_block_delta':
            delta = chunk['delta'].get('text', '')
            full_text += delta
            print(delta, end='', flush=True)

        elif kind == 'message_delta':
            output_toks = chunk.get('usage', {}).get('output_tokens', 0)

        elif kind == 'message_start':
            input_toks = chunk.get('message', {}).get('usage', {}).get('input_tokens', 0)

    print(f"\n\n[end_turn | in:{input_toks} out:{output_toks} tokens]")
    return full_text

In [9]:
_ = stream(
    'Write a short poem about distributed systems — four stanzas, four lines each.',
    system='You are a poet who specialises in technical subjects.',
)

#

 Distributed Systems



The

 nodes

 aw

...

—
Ensures

 no

 single point of crack

.



[end_turn | in:36 out:157 tokens]


## 5 — Tool Use / Function Calling

Tools let the model call Python functions when it needs to. The loop is:

```
send message + tool definitions
    ↓
model returns stop_reason='tool_use' + tool call(s)
    ↓
we execute the tool locally
    ↓
send tool result(s) back
    ↓
model returns final answer (stop_reason='end_turn')
```

### 5.1 — Define the tools

In [9]:
import datetime
import math

# ── Tool implementations ───────────────────────────────────────────────────────

def get_current_date() -> str:
    return datetime.date.today().isoformat()


def calculate(expression: str) -> str:
    """
    Evaluate a safe mathematical expression.
    Allows: numbers, +, -, *, /, **, (), and math.* functions.
    """
    allowed = {k: getattr(math, k) for k in dir(math) if not k.startswith('_')}
    allowed['abs'] = abs
    try:
        result = eval(expression, {'__builtins__': {}}, allowed)  # noqa: S307
        return str(result)
    except Exception as e:
        return f'Error: {e}'


# ── Tool registry ──────────────────────────────────────────────────────────────

TOOL_REGISTRY = {
    'get_current_date': lambda **_: get_current_date(),
    'calculate':        lambda expression, **_: calculate(expression),
}

# ── Tool definitions (sent to the model) ──────────────────────────────────────

TOOLS = [
    {
        'name': 'get_current_date',
        'description': 'Returns today\'s date in ISO 8601 format (YYYY-MM-DD).',
        'input_schema': {
            'type': 'object',
            'properties': {},
            'required': [],
        },
    },
    {
        'name': 'calculate',
        'description': (
            'Evaluates a mathematical expression and returns the result as a string. '
            'Supports standard arithmetic, exponentiation, and math functions '
            '(sqrt, sin, cos, log, etc.). Example: "sqrt(144) + 2**8"'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'expression': {
                    'type': 'string',
                    'description': 'A valid Python mathematical expression to evaluate.',
                },
            },
            'required': ['expression'],
        },
    },
]

print('Tools defined:', [t['name'] for t in TOOLS])

Tools defined: ['get_current_date', 'calculate']


### 5.2 — The tool loop

In [10]:
def invoke_with_tools(user_message, tools=TOOLS, system=None, max_tokens=1024, verbose=True):
    """
    Run a full tool-use loop.
    Keeps calling the model until stop_reason is 'end_turn' (no more tool calls).
    Returns the final text reply.
    """
    messages = [{'role': 'user', 'content': user_message}]

    while True:
        body = {
            'anthropic_version': ANT_VER,
            'messages': messages,
            'max_tokens': max_tokens,
            'tools': tools,
        }
        if system:
            body['system'] = system

        response = json.loads(client.invoke_model(
            modelId=MODEL_ID,
            body=json.dumps(body),
        )['body'].read())

        stop_reason = response['stop_reason']
        content     = response['content']

        # Append the assistant turn
        messages.append({'role': 'assistant', 'content': content})

        if stop_reason == 'end_turn':
            # Extract the final text block
            for block in content:
                if block.get('type') == 'text':
                    return block['text']
            return ''

        if stop_reason != 'tool_use':
            raise RuntimeError(f'Unexpected stop_reason: {stop_reason}')

        # Execute each tool call and collect results
        tool_results = []
        for block in content:
            if block.get('type') != 'tool_use':
                continue
            tool_id   = block['id']
            tool_name = block['name']
            tool_input = block.get('input', {})

            if verbose:
                print(f'  → tool call: {tool_name}({tool_input})')

            if tool_name not in TOOL_REGISTRY:
                result = f'Error: unknown tool "{tool_name}"'
            else:
                result = TOOL_REGISTRY[tool_name](**tool_input)

            if verbose:
                print(f'  ← result:    {result}')

            tool_results.append({
                'type':        'tool_result',
                'tool_use_id': tool_id,
                'content':     result,
            })

        # Feed results back in the next user turn
        messages.append({'role': 'user', 'content': tool_results})

### 5.3 — Try it out

In [11]:
# Should call get_current_date
reply = invoke_with_tools("What's today's date?")
print('\nFinal reply:', reply)

  → tool call: get_current_date({})
  ← result:    2026-06-16

Final reply: Today's date is **June 16, 2026**. Is there anything else I can help you with?


In [12]:
# Should call calculate
reply = invoke_with_tools('What is the square root of 1764, multiplied by the cosine of 0?')
print('\nFinal reply:', reply)

  → tool call: calculate({'expression': 'sqrt(1764)'})
  ← result:    42.0
  → tool call: calculate({'expression': 'cos(0)'})
  ← result:    1.0
  → tool call: calculate({'expression': '42.0 * 1.0'})
  ← result:    42.0

Final reply: Here's the breakdown:

- **√1764 = 42**
- **cos(0) = 1**
- **42 × 1 = 42**

The square root of 1764, multiplied by the cosine of 0, is **42**! Since cos(0) = 1, it doesn't change the value of the square root.


In [13]:
# Should call both tools in one turn
reply = invoke_with_tools(
    "What year is it, and what is 2 raised to the power of that year's last two digits?"
)
print('\nFinal reply:', reply)

  → tool call: get_current_date({})
  ← result:    2026-06-16
  → tool call: calculate({'expression': '2**26'})
  ← result:    67108864

Final reply: Here are the results:

- **Current year:** 2026
- **Last two digits:** 26
- **2²⁶ =** 67,108,864


## 6 — `BedrockChat` Utility Class

Wraps everything above into a reusable class:
- Maintains conversation history across turns
- Toggle streaming on/off
- Optional tools with automatic loop handling
- `new_session()` to reset history

In [14]:
class BedrockChat:
    """
    Stateful conversational wrapper around bedrock-runtime.

    Handles:
    - Multi-turn history
    - Optional system prompt
    - Streaming or blocking response modes
    - Tool-use loop (when tools are provided)
    """

    def __init__(
        self,
        client,
        model_id:    str  = MODEL_ID,
        system:      str  = None,
        streaming:   bool = False,
        tools:       list = None,
        max_tokens:  int  = 1024,
    ):
        self.client     = client
        self.model_id   = model_id
        self.system     = system
        self.streaming  = streaming
        self.tools      = tools
        self.max_tokens = max_tokens
        self.history    = []

    # ── Public ────────────────────────────────────────────────────────────────

    def chat(self, message: str) -> str:
        """Send a message, get a reply. History is updated automatically."""
        self.history.append({'role': 'user', 'content': message})

        if self.tools:
            reply = self._tool_loop()
        elif self.streaming:
            reply = self._stream_turn()
        else:
            reply = self._blocking_turn()

        self.history.append({'role': 'assistant', 'content': reply})
        return reply

    def new_session(self):
        """Clear conversation history."""
        self.history = []
        print('History cleared.')

    def show_history(self):
        for m in self.history:
            role    = m['role'].upper()
            content = m['content']
            if isinstance(content, list):
                content = json.dumps(content)[:200]
            snippet = content[:120] + '...' if len(content) > 120 else content
            print(f'[{role}] {snippet}')

    # ── Internal ──────────────────────────────────────────────────────────────

    def _base_body(self):
        body = {
            'anthropic_version': ANT_VER,
            'messages':          self.history,
            'max_tokens':        self.max_tokens,
        }
        if self.system:
            body['system'] = self.system
        if self.tools:
            body['tools'] = self.tools
        return body

    def _blocking_turn(self) -> str:
        response = json.loads(self.client.invoke_model(
            modelId=self.model_id,
            body=json.dumps(self._base_body()),
        )['body'].read())
        return response['content'][0]['text']

    def _stream_turn(self) -> str:
        response = self.client.invoke_model_with_response_stream(
            modelId=self.model_id,
            body=json.dumps(self._base_body()),
        )
        full_text = ''
        for event in response['body']:
            chunk = json.loads(event['chunk']['bytes'])
            if chunk.get('type') == 'content_block_delta':
                delta = chunk['delta'].get('text', '')
                full_text += delta
                print(delta, end='', flush=True)
        print()  # newline after stream
        return full_text

    def _tool_loop(self) -> str:
        """
        Run the tool-use loop until stop_reason is 'end_turn'.
        Intermediate tool calls and results are appended to history in place.
        """
        while True:
            response    = json.loads(self.client.invoke_model(
                modelId=self.model_id,
                body=json.dumps(self._base_body()),
            )['body'].read())
            stop_reason = response['stop_reason']
            content     = response['content']

            if stop_reason == 'end_turn':
                for block in content:
                    if block.get('type') == 'text':
                        return block['text']
                return ''

            if stop_reason != 'tool_use':
                raise RuntimeError(f'Unexpected stop_reason: {stop_reason}')

            # Append assistant tool-use turn, execute tools, append results
            self.history.append({'role': 'assistant', 'content': content})

            tool_results = []
            for block in content:
                if block.get('type') != 'tool_use':
                    continue
                tool_name  = block['name']
                tool_input = block.get('input', {})
                print(f'  → {tool_name}({tool_input})')
                result = (
                    TOOL_REGISTRY[tool_name](**tool_input)
                    if tool_name in TOOL_REGISTRY
                    else f'Error: unknown tool "{tool_name}"'
                )
                print(f'  ← {result}')
                tool_results.append({
                    'type':        'tool_result',
                    'tool_use_id': block['id'],
                    'content':     result,
                })

            self.history.append({'role': 'user', 'content': tool_results})

### 6.1 — Basic chat with history

In [15]:
bot = BedrockChat(client, system='You are a concise assistant. Answer in 1-2 sentences.')

for question in [
    'What is the CAP theorem?',
    'Which of the three can you actually drop in practice?',
    'Give me a one-line summary of what we just discussed.',
]:
    print(f'You:   {question}')
    print(f'Bot:   {bot.chat(question)}\n')

You:   What is the CAP theorem?
Bot:   The CAP theorem states that a distributed system can only guarantee **two out of three** properties simultaneously: **Consistency** (all nodes see the same data), **Availability** (every request gets a response), and **Partition Tolerance** (the system continues operating despite network failures). Since network partitions are unavoidable in practice, systems must choose between consistency and availability during a partition.

You:   Which of the three can you actually drop in practice?
Bot:   In practice, you **cannot drop Partition Tolerance** — network failures are inevitable in any real distributed system. So the real trade-off is always between **Consistency and Availability** when a partition occurs.

You:   Give me a one-line summary of what we just discussed.
Bot:   The CAP theorem forces distributed systems to choose between Consistency and Availability during network partitions, since Partition Tolerance cannot be sacrificed in practice

### 6.2 — Streaming mode

In [16]:
stream_bot = BedrockChat(
    client,
    system='You are a storyteller.',
    streaming=True,
    max_tokens=300,
)

_ = stream_bot.chat('Tell me a two-paragraph story about a developer who discovers their tests pass for the wrong reasons.')

# The Green Lie

Marcus had been staring at the same authentication module for three days, convinced there was a subtle bug lurking somewhere in the token validation logic. When he finally ran the test suite and watched all forty-two tests bloom into satisfying green checkmarks, he leaned back in his chair and exhaled the kind of breath a person holds for seventy-two hours. He pushed his changes, closed his laptop, and slept better than he had in weeks. The code was good. The tests had spoken.

It wasn't until his colleague Priya was onboarding and reading through the test file to understand the system that she sent the quiet, devastating message: *"Hey, did you know the mock for the token validator is just returning* `true` *for everything? Like... literally everything."* Marcus opened the file and there it was — a `jest.mock()` call from eight months ago, written by a developer who had long since left the company, that had been silently short-circuiting every single assertion. The te

### 6.3 — Tool use via the class

In [18]:
tool_bot = BedrockChat(client, tools=TOOLS)

reply = tool_bot.chat("What's today's date and what is 365 multiplied by the day-of-month?")
print('\nFinal reply:', reply)

  → get_current_date({})
  ← 2026-06-12


  → calculate({'expression': '365 * 12'})
  ← 4380



Final reply: Here are the results:

- 📅 **Today's date:** June 12, 2026
- 🔢 **365 × 12 (the day-of-month) = 4,380**


In [19]:
# Inspect full history including tool calls and results
tool_bot.show_history()

[USER] What's today's date and what is 365 multiplied by the day-of-month?
[ASSISTANT] [{"type": "text", "text": "Let me grab today's date first, and then I'll use it to calculate the multiplication!"}, {"ty...
[USER] [{"type": "tool_result", "tool_use_id": "toolu_bdrk_01DxBC6tUBrSxcf71NVs9Rpy", "content": "2026-06-12"}]
[ASSISTANT] [{"type": "text", "text": "Today is **June 12, 2026**, so the day-of-month is **12**. Now let me calculate 365 \u00d7 12...
[USER] [{"type": "tool_result", "tool_use_id": "toolu_bdrk_01LS2CAWWRK9PNjgGjWMS83D", "content": "4380"}]
[ASSISTANT] Here are the results:

- 📅 **Today's date:** June 12, 2026
- 🔢 **365 × 12 (the day-of-month) = 4,380**


## 7 — Guardrails

Guardrails enforce content policies on model inputs and outputs — blocking topics, filtering words, detecting PII, and more. They operate independently of the model itself.

**Our guardrail** (`32labz8mu0fe` v1) is configured with:
- **Blocked topics:** competitors (Walmart, Target, Gap, etc.), Internal Customer IDs
- **Blocked words:** `cryptocurrency`, `bitcoin`, `crypto`
- **Content filters:** hate/insults/sexual → HIGH; violence/misconduct → MEDIUM; prompt attack → HIGH (input only)
- **Blocked message:** "I am not allowed to answer this question per my company's policy."

**Two ways to use guardrails:**

| API | What it does |
|---|---|
| `apply_guardrail` | Test guardrail logic against any text — no model invocation |
| `invoke_model` + guardrail params | Apply guardrail inline during inference |

| Section | Topic |
|---|---|
| 7.1 | `apply_guardrail` — standalone testing |
| 7.2 | `invoke_model` with guardrail parameters |
| 7.3 | Reading the guardrail trace |

In [17]:
GUARDRAIL_ID      = '32labz8mu0fe'
GUARDRAIL_VERSION = '1'

print(f'Guardrail: {GUARDRAIL_ID} v{GUARDRAIL_VERSION}')

Guardrail: 32labz8mu0fe v1


### 7.1 — `apply_guardrail` — Standalone Testing

`apply_guardrail` tests your guardrail against arbitrary text **without invoking a model**. Useful for:
- Understanding exactly what gets blocked and why
- Pre-flight checks before feeding text to the model
- Testing guardrail configuration changes

The `source` parameter tells the guardrail whether to treat the text as input (a user message) or output (a model response) — some policies (like prompt-attack detection) only apply to one side.

The response has two key fields:
- `action`: `'NONE'` (passed) or `'GUARDRAIL_INTERVENED'` (blocked)
- `assessments`: which policies triggered and how

In [18]:
def check_guardrail(text, source='INPUT'):
    """Run text through the guardrail without calling the model."""
    return client.apply_guardrail(
        guardrailIdentifier=GUARDRAIL_ID,
        guardrailVersion=GUARDRAIL_VERSION,
        source=source,
        content=[{'text': {'text': text}}],
    )


def show_assessment(response):
    """Pretty-print the guardrail assessment."""
    action = response.get('action', 'UNKNOWN')
    icon   = '✅' if action == 'NONE' else '🚫'
    print(f'{icon} Action: {action}')

    for assessment in response.get('assessments', []):
        if policy := assessment.get('topicPolicy'):
            for topic in policy.get('topics', []):
                print(f'   topic policy: "{topic["name"]}" → {topic["action"]}')

        if policy := assessment.get('contentPolicy'):
            for f in policy.get('filters', []):
                if f.get('action') == 'BLOCKED':
                    print(f'   content filter: {f["type"]} (confidence: {f["confidence"]}) → BLOCKED')

        if policy := assessment.get('wordPolicy'):
            for word in policy.get('customWords', []):
                if word.get('action') == 'BLOCKED':
                    print(f'   custom word: "{word["match"]}" → BLOCKED')
            for entry in policy.get('managedWordLists', []):
                if entry.get('action') == 'BLOCKED':
                    print(f'   managed word list: "{entry["match"]}" → BLOCKED')


# ── Test 1: Safe input ────────────────────────────────────────────────────────
print('Test 1: Safe input')
r = check_guardrail('Tell me about the history of the Roman Empire.')
show_assessment(r)

# ── Test 2: Blocked topic — competitor mention ────────────────────────────────
print('\nTest 2: Competitor topic')
r = check_guardrail('What is the return policy at Walmart?')
show_assessment(r)

# ── Test 3: Blocked word ──────────────────────────────────────────────────────
print('\nTest 3: Blocked word')
r = check_guardrail('What is bitcoin and how does it work?')
show_assessment(r)

Test 1: Safe input
✅ Action: NONE

Test 2: Competitor topic
🚫 Action: GUARDRAIL_INTERVENED
   topic policy: "Competitors" → BLOCKED

Test 3: Blocked word
🚫 Action: GUARDRAIL_INTERVENED
   custom word: "bitcoin" → BLOCKED


### 7.2 — `invoke_model` with Guardrail Parameters

Pass `guardrailIdentifier` and `guardrailVersion` directly to `invoke_model`. The guardrail evaluates in two phases:

1. **Input phase** — before the model runs. If blocked, the model is never called.
2. **Output phase** — after the model responds. If blocked, the model's response is replaced.

When a guardrail intervenes, the response body still has the normal Anthropic message structure, but the `content` contains the configured **blocked message** instead of the model's reply.

Detect intervention via the HTTP response header `x-amzn-bedrock-guardrail-action` (value: `'INTERVENED'`).

In [ ]:
def invoke_guarded(messages, system=None, max_tokens=1024):
    """invoke_model with guardrail applied. Returns (response_body, was_blocked)."""
    body = {
        'anthropic_version': ANT_VER,
        'messages': messages,
        'max_tokens': max_tokens,
    }
    if system:
        body['system'] = system

    response = client.invoke_model(
        modelId=MODEL_ID,
        body=json.dumps(body),
        guardrailIdentifier=GUARDRAIL_ID,
        guardrailVersion=GUARDRAIL_VERSION,
    )
    data             = json.loads(response['body'].read())
    guardrail_action = response['ResponseMetadata']['HTTPHeaders'].get(
        'x-amzn-bedrock-guardrail-action', 'NONE'
    )
    was_blocked = guardrail_action == 'INTERVENED'
    return data, was_blocked


def print_guarded_response(data, was_blocked):
    # When guardrail blocks at input phase, usage and stop_reason are absent
    # (the model was never called, so no tokens were processed)
    text     = data['content'][0]['text']
    usage    = data.get('usage', {})
    stop     = data.get('stop_reason', '-')
    in_toks  = usage.get('input_tokens', '-')
    out_toks = usage.get('output_tokens', '-')
    icon     = '🚫 BLOCKED' if was_blocked else '✅ PASSED'
    print(f'[{icon}]')
    print(text)
    print(f'[stop_reason: {stop} | in:{in_toks} out:{out_toks} tokens]')


# ── Example 1: Safe prompt — passes guardrail ─────────────────────────────────
print('=== Safe prompt ===')
data, blocked = invoke_guarded([
    {'role': 'user', 'content': 'What is machine learning? Answer in 2 sentences.'}
])
print_guarded_response(data, blocked)

# ── Example 2: Blocked topic ──────────────────────────────────────────────────
print('\n=== Competitor topic ===')
data, blocked = invoke_guarded([
    {'role': 'user', 'content': 'What products does Target sell in their clothing section?'}
])
print_guarded_response(data, blocked)

# ── Example 3: Blocked word ───────────────────────────────────────────────────
print('\n=== Blocked word ===')
data, blocked = invoke_guarded([
    {'role': 'user', 'content': 'Explain how cryptocurrency mining works.'}
])
print_guarded_response(data, blocked)

### 7.3 — Reading the Guardrail Trace

Pass `trace='ENABLED'` to `invoke_model` to include the full guardrail evaluation in the response body under the `amazon-bedrock-trace` key.

The trace shows every policy evaluated — including ones that **didn't** block — split into:
- `inputAssessment` — what was evaluated against the user's message
- `outputAssessments` — what was evaluated against the model's response

This is the primary tool for debugging guardrail behaviour: understanding why something was (or wasn't) blocked.

In [ ]:
def invoke_guarded_with_trace(user_message, max_tokens=512):
    """invoke_model with guardrail and trace enabled."""
    body = {
        'anthropic_version': ANT_VER,
        'messages': [{'role': 'user', 'content': user_message}],
        'max_tokens': max_tokens,
    }
    response = client.invoke_model(
        modelId=MODEL_ID,
        body=json.dumps(body),
        guardrailIdentifier=GUARDRAIL_ID,
        guardrailVersion=GUARDRAIL_VERSION,
        trace='ENABLED',
    )
    return json.loads(response['body'].read())


def show_trace(data):
    """Print the guardrail trace from a response body."""
    trace = data.get('amazon-bedrock-trace', {}).get('guardrail', {})
    if not trace:
        print('  (no trace data)')
        return

    def print_assessment(assessment, label):
        print(f'  [{label}]')
        if policy := assessment.get('topicPolicy'):
            for t in policy.get('topics', []):
                print(f'    topic: "{t["name"]}" → {t["action"]}')
        if policy := assessment.get('contentPolicy'):
            for f in policy.get('filters', []):
                print(f'    content: {f["type"]} confidence={f["confidence"]} → {f["action"]}')
        if policy := assessment.get('wordPolicy'):
            for w in policy.get('customWords', []):
                print(f'    custom word: "{w["match"]}" → {w["action"]}')

    for _, assessment in trace.get('inputAssessment', {}).items():
        print_assessment(assessment, 'INPUT')

    for _, assessments in trace.get('outputAssessments', {}).items():
        for assessment in assessments:
            print_assessment(assessment, 'OUTPUT')


# ── Blocked request: competitor topic ────────────────────────────────────────
print('=== Competitor topic ===')
data = invoke_guarded_with_trace('What are the best products at Gap this season?')
print(f'Response: {data["content"][0]["text"]}')
print('Trace:')
show_trace(data)

# ── Clean request ─────────────────────────────────────────────────────────────
print('\n=== Safe request ===')
data = invoke_guarded_with_trace('What is Python? One sentence.')
print(f'Response: {data["content"][0]["text"]}')
print('Trace:')
show_trace(data)